# Bike Rental Agent - Workshop Walkthrough

This notebook walks through building a LangChain bike rental agent with tool calls:
1. **State A**: LangChain agent with 6 tools (prices, availability, booking, etc.)
2. **State B**: Simulation with FutureAGI Simulate SDK
3. **Evaluation**: Using function calling and hallucination detection evals
4. **Optimization**: Improving the prompt with ProTeGi (textual gradients)

In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

## State A: LangChain Agent with Tools

We define 6 tools for the bike rental shop and create a LangChain agent.

In [ ]:
from bike_rental.tools import ALL_TOOLS, VEHICLES

print(f"Tools available: {[t.name for t in ALL_TOOLS]}")
print(f"\nVehicles in inventory: {len(VEHICLES)}")
for v in VEHICLES.values():
    status = 'Available' if v['available'] else 'Unavailable'
    print(f"  {v['name']} ({v['id']}): ${v['price_per_hour']}/hr - {status}")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_tool_calling_agent, AgentExecutor
from config import get_langchain_llm

# Deliberately suboptimal prompt
SYSTEM_PROMPT = (
    "You are a bike rental assistant. Help customers with bike rentals. "
    "You have access to tools for checking prices, availability, and making bookings."
)

llm = get_langchain_llm(temperature=0.7)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, ALL_TOOLS, prompt)
executor = AgentExecutor(agent=agent, tools=ALL_TOOLS, verbose=True)

In [ ]:
# Test: does the agent use tools correctly?
result = executor.invoke({"input": "What's the cheapest bike you have?"})
print(f"\nResponse: {result['output']}")

In [ ]:
# Test: does it confirm before booking? (it shouldn't with current prompt)
result = executor.invoke({"input": "Book me the road bike for tomorrow, name is Mika, 3 hours"})
print(f"\nResponse: {result['output']}")
print("\n--- Did it ask for confirmation before booking? ---")

In [ ]:
# Test: what happens with unavailable vehicle?
result = executor.invoke({"input": "I want the e-bike for Saturday"})
print(f"\nResponse: {result['output']}")
print("\n--- Did it suggest alternatives? ---")

## State B: Simulate with FutureAGI

Wrap the agent with `LangChainAgentWrapper` and run automated persona tests.

In [ ]:
from fi.simulate import LangChainAgentWrapper, TestRunner, Scenario, Persona

wrapper = LangChainAgentWrapper(agent=executor, system_prompt=SYSTEM_PROMPT)

scenario = Scenario(
    name="bike-rental-notebook",
    dataset=[
        Persona(
            persona={"name": "Sofia", "mood": "budget-conscious"},
            situation="Wants cheapest bike for 3 hours.",
            outcome="Finds cheapest option, shows price, books after confirmation.",
        ),
        Persona(
            persona={"name": "Tom", "mood": "confused"},
            situation="Never rented a bike, wants to ride in the park.",
            outcome="Gets guided to City Cruiser with clear explanation.",
        ),
    ],
)

runner = TestRunner()
report = await runner.run_test(
    run_test_name="bike-rental-notebook",
    agent_callback=wrapper,
    scenario=scenario,
)

for r in report.results:
    print(f"\n--- {r.persona.persona['name']} ---")
    print(r.transcript[:500])

## Evaluation

In [ ]:
from fi.simulate import evaluate_report

report = evaluate_report(
    report,
    eval_templates=["task_completion", "evaluate_function_calling", "is_helpful", "detect_hallucination"],
    model_name="turing_flash",
)

for r in report.results:
    name = r.persona.persona["name"]
    print(f"\n--- {name} ---")
    if r.evaluation:
        for tmpl, scores in r.evaluation.items():
            print(f"  {tmpl}: {scores.get('score', 'N/A')} - {scores.get('reason', '')[:150]}")

## Optimization with ProTeGi

ProTeGi uses **beam search + textual gradients**. It identifies specific failure patterns
(wrong tool calls, missing confirmations) and generates targeted fixes.

In [ ]:
from config import get_litellm_model
from fi.opt.generators import LiteLLMGenerator
from fi.opt.optimizers import ProTeGi
from fi.opt.base.evaluator import Evaluator
from fi.opt.datamappers import BasicDataMapper
from fi.evals.metrics import CustomLLMJudge
from fi.evals.llm import LiteLLMProvider

model = get_litellm_model()
teacher = LiteLLMGenerator(model=model, prompt_template="{prompt}")

judge = CustomLLMJudge(
    provider=LiteLLMProvider(),
    config={"name": "bike_judge", "grading_criteria": "Score 0-1: tool usage, confirmation before booking, recommendation quality, helpfulness."},
    model=model,
)

opt_evaluator = Evaluator(metric=judge)
mapper = BasicDataMapper(key_map={"response": "generated_output", "expected_response": "answer"})

dataset = [
    {"customer_message": "Cheapest bike for 3 hours?", "answer": "City Cruiser at $6/hr = $18. Want me to check availability?"},
    {"customer_message": "Book me the road bike tomorrow, I'm Mika", "answer": "Before booking: Road Bike Elite for Mika, tomorrow. How many hours?"},
    {"customer_message": "I want the e-bike", "answer": "Sorry, e-bike is unavailable. Try City Cruiser ($6/hr) or Mountain Bike ($8/hr) instead."},
]

initial_prompt = SYSTEM_PROMPT + "\n\nCustomer message: {customer_message}"

optimizer = ProTeGi(teacher_generator=teacher, num_gradients=3, beam_size=3)
result = optimizer.optimize(
    evaluator=opt_evaluator, data_mapper=mapper, dataset=dataset,
    initial_prompts=[initial_prompt], num_rounds=2,
)

print(f"Score: {result.final_score:.4f}")
print(f"\nOptimized prompt:\n{result.best_generator.get_prompt_template()}")